In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.shuffle.useOldFetchProtocol','true'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
#using hotels data

In [3]:
hotels_schema = "booking_id long, guest_name string, checking_date date, checkout_date date, room_type string, total_price float"

In [4]:
hotels_df = spark.read \
.format("csv") \
.schema(hotels_schema) \
.load("/public/trendytech/datasets/hotel_data.csv")

In [5]:
hotels_df

booking_id,guest_name,checking_date,checkout_date,room_type,total_price
1,John Doe,2023-05-01,2023-05-05,Standard,400.0
2,Jane Smith,2023-05-02,2023-05-06,Deluxe,600.0
3,Mark Johnson,2023-05-03,2023-05-08,Standard,450.0
4,Sarah Wilson,2023-05-04,2023-05-07,Executive,750.0
5,Emily Brown,2023-05-06,2023-05-09,Deluxe,550.0
6,Michael Davis,2023-05-07,2023-05-10,Standard,400.0
7,Samantha Thompson,2023-05-08,2023-05-12,Deluxe,600.0
8,William Lee,2023-05-10,2023-05-13,Standard,450.0
9,Amanda Harris,2023-05-11,2023-05-16,Executive,750.0
10,David Rodriguez,2023-05-12,2023-05-15,Deluxe,550.0


In [6]:
hotels_df.printSchema()

root
 |-- booking_id: long (nullable = true)
 |-- guest_name: string (nullable = true)
 |-- checking_date: date (nullable = true)
 |-- checkout_date: date (nullable = true)
 |-- room_type: string (nullable = true)
 |-- total_price: float (nullable = true)



1 - What is the average stay duration for each room type

In [7]:
from pyspark.sql.functions import *

In [8]:
hotels_df = hotels_df.withColumn("stay_duration", datediff("checkout_date", "checking_date"))

In [9]:
hotels_df.groupBy("room_type").agg(round(avg("stay_duration"),0).alias("avg_stay_duration")).show()

+---------+-----------------+
|room_type|avg_stay_duration|
+---------+-----------------+
|Executive|              8.0|
|   Deluxe|              8.0|
| Standard|              8.0|
+---------+-----------------+



2 - Guest who stayed longest

In [10]:
hotels_df.orderBy(col("stay_duration").desc()).select("guest_name", "stay_duration").limit(1).show()

+----------------+-------------+
|      guest_name|stay_duration|
+----------------+-------------+
|Daniel Rodriguez|           10|
+----------------+-------------+



3 - No of booking per room type

In [11]:
hotels_df.groupBy("room_type").count().alias("booking_count").show()

+---------+-----+
|room_type|count|
+---------+-----+
|Executive|   20|
|   Deluxe|   43|
| Standard|   44|
+---------+-----+



4 - Repeat guest more than 2 booking

In [12]:
hotels_df.groupBy("guest_name").count().filter("count > 1").show()

+----------------+-----+
|      guest_name|count|
+----------------+-----+
|     James Smith|   10|
|  Robert Johnson|   10|
| Sophia Thompson|    9|
|     William Lee|   11|
|    Olivia Brown|    9|
|   Lily Anderson|    9|
|    Emily Harris|    7|
|   Michael Davis|   11|
|   Sophia Wilson|    9|
|Daniel Rodriguez|    8|
| David Rodriguez|    2|
+----------------+-----+



In [13]:
## there might be multiple guest_name with same name so using booking_id as a num_of_booking
hotels_df.groupBy("guest_name").agg(count("booking_id").alias("num_bookings")).filter("num_bookings > 1").show()

+----------------+------------+
|      guest_name|num_bookings|
+----------------+------------+
|     James Smith|          10|
|  Robert Johnson|          10|
| Sophia Thompson|           9|
|     William Lee|          11|
|    Olivia Brown|           9|
|   Lily Anderson|           9|
|    Emily Harris|           7|
|   Michael Davis|          11|
|   Sophia Wilson|           9|
|Daniel Rodriguez|           8|
| David Rodriguez|           2|
+----------------+------------+



5 - Highest average revenue per booking by room_type

In [14]:
hotels_df.groupBy("room_type").agg(round(avg("total_price"),0).alias("avg_revenue")).orderBy(desc("avg_revenue")).show()

+---------+-----------+
|room_type|avg_revenue|
+---------+-----------+
|Executive|      750.0|
|   Deluxe|      576.0|
| Standard|      425.0|
+---------+-----------+



6 - Total Revenue by room type

In [15]:
hotels_df.groupBy("room_type").agg(sum("total_price").alias("total_revenue")).show()

+---------+-------------+
|room_type|total_revenue|
+---------+-------------+
|Executive|      15000.0|
|   Deluxe|      24750.0|
| Standard|      18700.0|
+---------+-------------+



7 - Bookings that overlap

In [16]:
hotels_df.alias("a").join(
    hotels_df.alias("b"),
    (col("a.booking_id") != col("b.booking_id")) &
    (col("a.checking_date") < col("b.checkout_date")) &
    (col("a.checkout_date") > col("b.checking_date"))
).select(
    col("a.booking_id").alias("booking_a"),
    col("b.booking_id").alias("booking_b"),
    col("a.guest_name").alias("guest_a"),
    col("b.guest_name").alias("guest_b")
).show()

+---------+---------+-------------+-----------------+
|booking_a|booking_b|      guest_a|          guest_b|
+---------+---------+-------------+-----------------+
|        1|        2|     John Doe|       Jane Smith|
|        1|        3|     John Doe|     Mark Johnson|
|        1|        4|     John Doe|     Sarah Wilson|
|        2|        1|   Jane Smith|         John Doe|
|        2|        3|   Jane Smith|     Mark Johnson|
|        2|        4|   Jane Smith|     Sarah Wilson|
|        3|        1| Mark Johnson|         John Doe|
|        3|        2| Mark Johnson|       Jane Smith|
|        3|        4| Mark Johnson|     Sarah Wilson|
|        3|        5| Mark Johnson|      Emily Brown|
|        3|        6| Mark Johnson|    Michael Davis|
|        4|        1| Sarah Wilson|         John Doe|
|        4|        2| Sarah Wilson|       Jane Smith|
|        4|        3| Sarah Wilson|     Mark Johnson|
|        4|        5| Sarah Wilson|      Emily Brown|
|        5|        3|  Emily

finding cases where two different bookings have overlapping stays, which could indicate potential duplicate reservations, room-sharing situations, or booking conflicts.

8 - Most popular check in date

In [19]:
hotels_df.groupBy("checking_date").count().orderBy(desc("count")).show(1)

+-------------+-----+
|checking_date|count|
+-------------+-----+
|   2023-07-29|    2|
+-------------+-----+
only showing top 1 row



9 - Guest who stayed during a holiday week May 1 - May 7 2023

In [20]:
holiday_start = to_date(lit("2023-05-01"))
holiday_end = to_date(lit("2023-05-07"))

In [21]:
hotels_df.filter(
(col('checking_date') <= holiday_end) &
    (col('checkout_date') >= holiday_start)
).select("guest_name", "checking_date","checkout_date").show()

+-------------+-------------+-------------+
|   guest_name|checking_date|checkout_date|
+-------------+-------------+-------------+
|     John Doe|   2023-05-01|   2023-05-05|
|   Jane Smith|   2023-05-02|   2023-05-06|
| Mark Johnson|   2023-05-03|   2023-05-08|
| Sarah Wilson|   2023-05-04|   2023-05-07|
|  Emily Brown|   2023-05-06|   2023-05-09|
|Michael Davis|   2023-05-07|   2023-05-10|
+-------------+-------------+-------------+



In [24]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank, datediff

10 - Top 3 longest stays for each room type

In [25]:
window_spec = Window.partitionBy("room_type").orderBy(datediff("checkout_date", "checking_date").desc())

In [28]:
hotels_df.withColumn("stay_duration", datediff("checkout_date", "checking_date")) \
  .withColumn("rank", rank().over(window_spec)) \
  .filter(col("rank") <= 3) \
  .select("guest_name", "room_type", "stay_duration", "rank") \
  .orderBy("room_type", "rank") \
  .show()

+----------------+---------+-------------+----+
|      guest_name|room_type|stay_duration|rank|
+----------------+---------+-------------+----+
|  Robert Johnson|   Deluxe|           10|   1|
|    Olivia Brown|   Deluxe|           10|   1|
|Daniel Rodriguez|   Deluxe|           10|   1|
|Daniel Rodriguez|   Deluxe|           10|   1|
| Sophia Thompson|   Deluxe|           10|   1|
|    Olivia Brown|   Deluxe|           10|   1|
|  Robert Johnson|   Deluxe|           10|   1|
|Daniel Rodriguez|   Deluxe|           10|   1|
| Sophia Thompson|   Deluxe|           10|   1|
|    Olivia Brown|   Deluxe|           10|   1|
|  Robert Johnson|   Deluxe|           10|   1|
|Daniel Rodriguez|   Deluxe|           10|   1|
| Sophia Thompson|   Deluxe|           10|   1|
|    Olivia Brown|   Deluxe|           10|   1|
|  Robert Johnson|   Deluxe|           10|   1|
| Sophia Thompson|   Deluxe|           10|   1|
|    Emily Harris|Executive|           10|   1|
|     James Smith|Executive|           1

11 - . Find the first guest (chronologically) for each room type

In [30]:
window_spec = Window.partitionBy("room_type").orderBy("checking_date")

hotels_df.withColumn("row_num", row_number().over(window_spec)) \
  .filter(col("row_num") == 1) \
  .select("guest_name", "room_type", "checking_date") \
  .show()

+------------+---------+-------------+
|  guest_name|room_type|checking_date|
+------------+---------+-------------+
|Sarah Wilson|Executive|   2023-05-04|
|  Jane Smith|   Deluxe|   2023-05-02|
|    John Doe| Standard|   2023-05-01|
+------------+---------+-------------+



12 - Find the most recent booking for each guest

In [36]:
window_spec = Window.partitionBy("guest_name").orderBy(col("checking_date").desc())

In [37]:
hotels_df.withColumn("row_num", row_number().over(window_spec)) \
  .filter(col("row_num") == 1) \
  .select("guest_name", "checking_date", "room_type") \
  .show()

+-----------------+-------------+---------+
|       guest_name|checking_date|room_type|
+-----------------+-------------+---------+
|Samantha Thompson|   2023-05-08|   Deluxe|
|      Emily Brown|   2023-05-06|   Deluxe|
|     Mark Johnson|   2023-05-03| Standard|
|     Sarah Wilson|   2023-05-04|Executive|
|    Amanda Harris|   2023-05-11|Executive|
|      James Smith|   2023-08-14|Executive|
|   Robert Johnson|   2023-08-11|   Deluxe|
|  Sophia Thompson|   2023-08-17|   Deluxe|
|      William Lee|   2023-08-19| Standard|
|     Olivia Brown|   2023-08-15|   Deluxe|
|       Emma Brown|   2023-06-01|   Deluxe|
|    Lily Anderson|   2023-08-12| Standard|
|     Emily Harris|   2023-08-07|Executive|
|    Michael Davis|   2023-08-16| Standard|
|       Ava Harris|   2023-05-24|Executive|
|    Sophia Wilson|   2023-08-10| Standard|
|   Emily Thompson|   2023-05-21|   Deluxe|
|         John Doe|   2023-05-01| Standard|
|       Jane Smith|   2023-05-02|   Deluxe|
|     Linda Wilson|   2023-05-14

In [42]:
window_spec = Window.orderBy(col("total_price").desc())

hotels_df.withColumn("price_rank", dense_rank().over(window_spec)) \
  .select("guest_name", "room_type", "total_price", "price_rank") \
  .show(50)

+-----------------+---------+-----------+----------+
|       guest_name|room_type|total_price|price_rank|
+-----------------+---------+-----------+----------+
|     Sarah Wilson|Executive|      750.0|         1|
|    Amanda Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|       Ava Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|     Emily Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|     Emily Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|     Emily Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|     Emily Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|     Emily Harris|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|         1|
|      James Smith|Executive|      750.0|     

13 - Identify guests who made multiple bookings and rank them by stay duration per guest

In [43]:
hotels_df.withColumn("stay_duration", datediff("checkout_date", "checking_date")) \
  .withColumn("duration_rank", rank().over(window_spec)) \
  .filter(col("duration_rank") == 1) \
  .select("guest_name", "room_type", "stay_duration", "duration_rank") \
  .show()

+-------------+---------+-------------+-------------+
|   guest_name|room_type|stay_duration|duration_rank|
+-------------+---------+-------------+-------------+
| Sarah Wilson|Executive|            3|            1|
|Amanda Harris|Executive|            5|            1|
|  James Smith|Executive|            6|            1|
|   Ava Harris|Executive|            6|            1|
|  James Smith|Executive|            7|            1|
| Emily Harris|Executive|            7|            1|
|  James Smith|Executive|            8|            1|
| Emily Harris|Executive|            9|            1|
|  James Smith|Executive|           10|            1|
| Emily Harris|Executive|           10|            1|
|  James Smith|Executive|           10|            1|
| Emily Harris|Executive|           10|            1|
|  James Smith|Executive|           10|            1|
| Emily Harris|Executive|            9|            1|
|  James Smith|Executive|            9|            1|
|  James Smith|Executive|   

In [17]:
#spark.stop()